# GPT

In [3]:
# part 1: 导入相关的 package
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from dataclasses import dataclass

import math
import json
torch.manual_seed(1024)

import tiktoken

class MyDataset(Dataset):
    def __init__(self, file_path, block_size) -> None:
        super().__init__()
        self.enc = tiktoken.get_encoding('gpt-2')
        self.spatial_code = self.enc.encode("<|endoftext|>",allowed_special={"<|endoftext|>"})[0]
        
        self.encode_list = []
        
        with open(file_path,'r') as f:
            
            for idx,line in enumerate(f):
                try:
                    line = json.loads(line)['text']
                    line_token_encode =  self.enc.encode(line) + [self.spatial_code]
                    self.encode_list.extend(line_token_encode)
                except Exception as e:
                    continue
        self.result_list = []
        for i in range(0,len(self.encode_list),block_size):
            if i+block_size>len(self.encode_list):
                self.result_list.append(self.encode_list[i:] + [self.spatial_code]*(i+block_size-len(self.encode_list)))
            else:
                self.result_list.append(self.encode_list[i:i+block_size])
        
    def __getitems__(self, index):
        
        current_item = self.result_list[index]
        x = current_item[:-1]
        y = current_item[1:]
        return x,y
    
    def __len__(self):
        return 0
    
    def encode(self):
        return 0
    
    def decode(self):
        return 0
    
    

In [4]:
import torch
torch.tril(torch.ones(5,5))

tensor([[1., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0.],
        [1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1.]])

In [ ]:
import torch
import torch.nn as nn

batch_size = 12
block_size = 512
head_dim = 128
class SingleHeadAttention(nn.Module):
    def __init__(self,) -> None:
        super().__init__()
        self.Q = nn.Linear(head_dim,head_dim)
        self.K = nn.Linear(head_dim,head_dim)
        self.V = nn.Linear(head_dim,head_dim)
        self.register_buffer('attention_mask',torch.tril(torch.ones(block_size,block_size)))
        
    def forward(self,x:torch.Tensor):
        q_value = self.Q(x)
        k_value = self.K(x)
        v_value = self.V(x)
        
        attention_weight = q_value @ k_value.transpose(-1,-2)
        masked_attention = torch.masked_fill(attention_weight,self.attention_mask,float('-inf'))
        
        attention_weight = F.softmax(masked_attention,dim=-1) / math.sqrt(head_dim)
        return attention_weight*v_value

In [16]:
t1 = torch.randn((3,3))
t2=torch.tril(torch.ones(3,3))
t2==0

tensor([[False,  True,  True],
        [False, False,  True],
        [False, False, False]])

In [17]:
torch.masked_fill(t1,t2==0,float('-inf'))

tensor([[-1.6357,    -inf,    -inf],
        [-1.6535,  1.8129,    -inf],
        [-0.7004,  0.8429,  0.8971]])